In [ ]:
import os

import pandas as pd
import plotly.express as px

from util import CensusApi

c = CensusApi(os.getenv("CENSUS_KEY"),timeout=90)

county_ids = [53033, 53035, 53053, 53061]
state_id = 53


def fetch_totals(cols_dict, year, dataset):
    """Pull a decennial table and return a DataFrame indexed by county geoid
    with one column per age group."""
    df = c.get_dec_data(cols_dict, year, 'county', dataset, county_ids, state_id)
    df = df.loc[df.geoid.isin(county_ids)].drop(columns=['name']).set_index('geoid')
    return df


def compute_headship_rates(year_config):
    """Given {year: {hh: (dataset, dict), pop: (dataset, dict)}}, return a
    DataFrame indexed by (county_id, age) with one column per year."""
    frames = []
    for year, cfg in year_config.items():
        hh_dataset, hh_cols = cfg['hh']
        pop_dataset, pop_cols = cfg['pop']
        hh = fetch_totals(hh_cols, year, hh_dataset)
        pop = fetch_totals(pop_cols, year, pop_dataset)
        rate = (hh / pop).stack().rename('headship_rate_' + str(year))
        frames.append(rate)
    out = pd.concat(frames, axis=1).sort_index(axis=1)
    out.index.names = ['county_id', 'age']
    return out

#### Decennial census variable specs

In [ ]:
# Population in households, broken down by sex and detailed age bins.
# Row positions are identical across all three tables; only the variable-name
# format differs.
#
#   2000 SF4  -> PCT005xxx
#   2010 SF1  -> PCT013xxx
#   2020 DHC  -> PCT13_xxxN
#
# Male bins are 003-025; female bins are 027-049 (same offsets in every table).

# 2000 SF4 PCT005: SEX BY AGE FOR THE POPULATION IN HOUSEHOLDS.
POP_DICT_SF4_2000 = {
    # 15-17 + 18-19 + 20 + 21 + 22-24
    'age_15_24': [
        'PCT005006', 'PCT005007', 'PCT005008', 'PCT005009', 'PCT005010',
        'PCT005030', 'PCT005031', 'PCT005032', 'PCT005033', 'PCT005034',
    ],
    # 25-29 + 30-34
    'age_25_34': ['PCT005011', 'PCT005012', 'PCT005035', 'PCT005036'],
    # 35-39 + 40-44
    'age_35_44': ['PCT005013', 'PCT005014', 'PCT005037', 'PCT005038'],
    # 45-49 + 50-54
    'age_45_54': ['PCT005015', 'PCT005016', 'PCT005039', 'PCT005040'],
    # 55-59 + 60-61 + 62-64
    'age_55_64': [
        'PCT005017', 'PCT005018', 'PCT005019',
        'PCT005041', 'PCT005042', 'PCT005043',
    ],
    # 65-66 + 67-69 + 70-74
    'age_65_74': [
        'PCT005020', 'PCT005021', 'PCT005022',
        'PCT005044', 'PCT005045', 'PCT005046',
    ],
    # 75-79 + 80-84
    'age_75_84': ['PCT005023', 'PCT005024', 'PCT005047', 'PCT005048'],
    # 85+
    'age_85_plus': ['PCT005025', 'PCT005049'],
}

# 2010 SF1 PCT013: SEX BY AGE FOR THE POPULATION IN HOUSEHOLDS.
POP_DICT_SF1_2010 = {
    'age_15_24': [
        'PCT013006', 'PCT013007', 'PCT013008', 'PCT013009', 'PCT013010',
        'PCT013030', 'PCT013031', 'PCT013032', 'PCT013033', 'PCT013034',
    ],
    'age_25_34': ['PCT013011', 'PCT013012', 'PCT013035', 'PCT013036'],
    'age_35_44': ['PCT013013', 'PCT013014', 'PCT013037', 'PCT013038'],
    'age_45_54': ['PCT013015', 'PCT013016', 'PCT013039', 'PCT013040'],
    'age_55_64': [
        'PCT013017', 'PCT013018', 'PCT013019',
        'PCT013041', 'PCT013042', 'PCT013043',
    ],
    'age_65_74': [
        'PCT013020', 'PCT013021', 'PCT013022',
        'PCT013044', 'PCT013045', 'PCT013046',
    ],
    'age_75_84': ['PCT013023', 'PCT013024', 'PCT013047', 'PCT013048'],
    'age_85_plus': ['PCT013025', 'PCT013049'],
}

# 2020 DHC PCT13: SEX BY AGE FOR THE POPULATION IN HOUSEHOLDS.
POP_DICT_DHC_2020 = {
    'age_15_24': [
        'PCT13_006N', 'PCT13_007N', 'PCT13_008N', 'PCT13_009N', 'PCT13_010N',
        'PCT13_030N', 'PCT13_031N', 'PCT13_032N', 'PCT13_033N', 'PCT13_034N',
    ],
    'age_25_34': ['PCT13_011N', 'PCT13_012N', 'PCT13_035N', 'PCT13_036N'],
    'age_35_44': ['PCT13_013N', 'PCT13_014N', 'PCT13_037N', 'PCT13_038N'],
    'age_45_54': ['PCT13_015N', 'PCT13_016N', 'PCT13_039N', 'PCT13_040N'],
    'age_55_64': [
        'PCT13_017N', 'PCT13_018N', 'PCT13_019N',
        'PCT13_041N', 'PCT13_042N', 'PCT13_043N',
    ],
    'age_65_74': [
        'PCT13_020N', 'PCT13_021N', 'PCT13_022N',
        'PCT13_044N', 'PCT13_045N', 'PCT13_046N',
    ],
    'age_75_84': ['PCT13_023N', 'PCT13_024N', 'PCT13_047N', 'PCT13_048N'],
    'age_85_plus': ['PCT13_025N', 'PCT13_049N'],
}

YEAR_CONFIG = {
    2000: {
        'hh': ('sf1', {
            'age_15_24': ['P021003', 'P021012'],
            'age_25_34': ['P021004', 'P021013'],
            'age_35_44': ['P021005', 'P021014'],
            'age_45_54': ['P021006', 'P021015'],
            'age_55_64': ['P021007', 'P021016'],
            'age_65_74': ['P021008', 'P021017'],
            'age_75_84': ['P021009', 'P021018'],
            'age_85_plus': ['P021010', 'P021019'],
        }),
        'pop': ('sf4', POP_DICT_SF4_2000),
    },
    2010: {
        'hh': ('sf1', {
            'age_15_24': ['P022003', 'P022013'],
            'age_25_34': ['P022004', 'P022014'],
            'age_35_44': ['P022005', 'P022015'],
            'age_45_54': ['P022006', 'P022016'],
            'age_55_64': ['P022007', 'P022017', 'P022008', 'P022018'],
            'age_65_74': ['P022009', 'P022019'],
            'age_75_84': ['P022010', 'P022020'],
            'age_85_plus': ['P022011', 'P022021'],
        }),
        'pop': ('sf1', POP_DICT_SF1_2010),
    },
    2020: {
        'hh': ('dhc', {
            'age_15_24': ['PCT3_003N', 'PCT3_013N'],
            'age_25_34': ['PCT3_004N', 'PCT3_014N'],
            'age_35_44': ['PCT3_005N', 'PCT3_015N'],
            'age_45_54': ['PCT3_006N', 'PCT3_016N'],
            'age_55_64': ['PCT3_007N', 'PCT3_017N', 'PCT3_008N', 'PCT3_018N'],
            'age_65_74': ['PCT3_009N', 'PCT3_019N'],
            'age_75_84': ['PCT3_010N', 'PCT3_020N'],
            'age_85_plus': ['PCT3_011N', 'PCT3_021N'],
        }),
        'pop': ('dhc', POP_DICT_DHC_2020),
    },
}

In [ ]:
headship_rates = compute_headship_rates(YEAR_CONFIG)

# Add a 0 headship rate for the under-15 age group for each county so the
# index aligns with the PUMS hhpop series (which includes 0-14).
zeros = pd.DataFrame(
    0.0,
    index=pd.MultiIndex.from_product(
        [county_ids, ['age_0_14']], names=headship_rates.index.names
    ),
    columns=headship_rates.columns,
)
headship_rates = pd.concat([headship_rates, zeros]).sort_index()
headship_rates

In [ ]:
pums = pd.read_csv('data/pums2023_hhpop.csv')
PUMS_AGE_MAP = {
    'ages_0_4': 'age_0_14', 'ages_5_9': 'age_0_14', 'ages_10_14': 'age_0_14',
    'ages_15_19': 'age_15_24', 'ages_20_24': 'age_15_24',
    'ages_25_29': 'age_25_34', 'ages_30_34': 'age_25_34',
    'ages_35_39': 'age_35_44', 'ages_40_44': 'age_35_44',
    'ages_45_49': 'age_45_54', 'ages_50_54': 'age_45_54',
    'ages_55_59': 'age_55_64', 'ages_60_64': 'age_55_64',
    'ages_65_69': 'age_65_74', 'ages_70_74': 'age_65_74',
    'ages_75_79': 'age_75_84', 'ages_80_84': 'age_75_84',
    'ages_85_plus': 'age_85_plus',
}

pums = (
    pums.assign(age=pums['age'].map(PUMS_AGE_MAP))
    .dropna(subset=['age'])
    .groupby(['county_id','age'])['hhpop'].sum()
)

In [ ]:
df = headship_rates.copy()
df['hhpop'] = pums
df['hh_2000'] = df['headship_rate_2000'] * df['hhpop']
df['hh_2010'] = df['headship_rate_2010'] * df['hhpop']
df['hh_2020'] = df['headship_rate_2020'] * df['hhpop']

In [ ]:
df.reset_index().to_csv('data/headship_rates_by_age.csv', index=False)

In [ ]:
remi = pd.read_csv('data/remi2060_hhpop.csv')
remi = (
    remi.assign(age=remi['age'].map(PUMS_AGE_MAP))
    .dropna(subset=['age'])
    .groupby(['county_id','age'])['hhpop'].sum()
)

In [ ]:
df = headship_rates.copy()
df['hhpop'] = remi
df['hh_2000'] = df['headship_rate_2000'] * df['hhpop']
df['hh_2010'] = df['headship_rate_2010'] * df['hhpop']
df['hh_2020'] = df['headship_rate_2020'] * df['hhpop']

In [ ]:
df.reset_index().to_csv('data/remi2060_headship_rates_by_age.csv', index=False)